# AMLC - agentic search loop + experiments

Code by [Doug Turnbull](http://softwaredoug.com) for Applied ML Conference.

This notebook has a basic agentic search loop

* We have a set of furniture in our catalog
* We tell the Agent our preferences
* The agent uses the search tool to recommend furniture
* Ported code from [this blog post](https://softwaredoug.com/blog/2025/10/06/how-much-does-reasoning-improve-search-quality)

## Adding other retrieval tools

Adding an embedding + lexical tool together seem to help

## Experimentation

Further experiments with harness design, hooks, skills

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)    # colab, share data across notebook runs on gdrive
# mount(use_gdrive=False) # <- colab without gdrive
# mount(use_gdrive=False, manual_path="/path/to/directory")  # <- force data path to specific directory, ie you're running locally.


  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-mlaik6do
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-mlaik6do
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 8d178163d60ae26dfb8eed0a738c5f4e9d643b7e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.3/745.3 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 71.0 MB/s eta 0:00:00
  Created wheel for cheat_at_search: filename=cheat_at_search-0.1.0-py3-none-any.whl size=1455059 sha256=ac76e2ec133a71f2bb955651ee8949798464d715c4c1e84566485a6e852c43e4
  Stored in directory: /tmp/pip-ephem-wheel-cache-fqu0fs64/wheels/ea/96/d3/e69453e5f86e6f891864e5b5baf2b11ffda2d09

## Get an OpenAI Key + load corpus

This will prompt you for an OpenAI Key to interact with GPT-5

In [ ]:
from cheat_at_search.data_dir import key_for_provider
from openai import OpenAI
from cheat_at_search.wands_data import corpus

OPENAI_KEY = key_for_provider("openai")

openai = OpenAI(api_key=OPENAI_KEY)

corpus['category'] = corpus['category'].str.strip()
corpus['sub_category'] = corpus['sub_category'].str.strip()

corpus

You're going to be prompted for your API key. This will be stored in a local file
If you'd prefer to set it as an environment variable, set it as:
    export OPENAI_API_KEY=your_api_key_here
Enter your openai_api_key: ··········


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathroom...
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,,basematerialdetails : steel| : gray wood|ofhar...,1314.0,4.5,864.0,"[basematerialdetails : steel, : gray wood, of...",42990,emmeline 5 piece breakfast dining set,,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42991,42991,maloney 3 piece pub table set,Dining Table Sets,Furniture / Kitchen & Dining Furniture / Dinin...,this pub table set includes 1 counter height t...,additionaltoolsrequirednotincluded : power dri...,49.0,4.0,41.0,[additionaltoolsrequirednotincluded : power dr...,42991,maloney 3 piece pub table set,this pub table set includes 1 counter height t...,Furniture,Kitchen & Dining Furniture,Furniture / Kitchen & Dining Furniture
42992,42992,fletcher 27.5 '' wide polyester armchair,Teen Lounge Furniture|Accent Chairs,Furniture / Living Room Furniture / Chairs & S...,"bring iconic , modern style to your space in a...",legmaterialdetails : rubberwood|backheight-sea...,1746.0,4.5,1226.0,"[legmateri

### Index the furniture

We'll index title and description with basic stemming to be able to retrieve them

In [ ]:
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer

corpus['title_snowball'] = SearchArray.index(corpus['title'].fillna(''), snowball_tokenizer)
corpus['description_snowball'] = SearchArray.index(corpus['description'].fillna(''), snowball_tokenizer)
corpus['category_snowball'] = SearchArray.index(corpus['category'].fillna(''), snowball_tokenizer)

2026-04-21 14:47:41,497 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-04-21 14:47:41,518 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-04-21 14:47:41,524 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

INFO:searcharray.indexing:Batch tokenization complete


2026-04-21 14:48:02,472 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-04-21 14:48:02,492 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete


### Setup search tools

Here is a function that searches a Wayfair product dataset. It's just a Python function that returns top 10 pieces of furniture.

It also allows the agent to filter by category. **Notice** this inches towards the metadata style similarity.

In [ ]:
import numpy as np
from typing import Union

from pydantic import BaseModel, Field
from typing import Optional, Literal


Categories = Literal['Furniture', 'Kitchen & Tabletop', 'Browse By Brand',
                     'Home Improvement', 'Décor & Pillows', 'Outdoor',
                     'Storage & Organization', 'Bed & Bath', 'Baby & Kids',
                     'Pet', 'Lighting', 'Rugs', 'School Furniture and Supplies',
                     'Commercial Business Furniture', 'Holiday Décor', 'Fountains',
                     'Contractor', 'Appliances', 'Sale', 'Reception Area',
                     'Foodservice', 'Institutional Furniture Parts & Accessories',
                     'Landscaping Screens & Bridges', 'Shop Product Type', 'Clips',
                     'Slicers, Peelers And Graters', 'Bed Accessories',
                     'Accommodations', 'Buffet Accessories', 'Specialty Serving',
                     'Display Cases', 'Key Organizers', 'Ergonomic Accessories',
                     'Slow Cookers', 'Bath Rugs & Mats', 'Furniture Cushions',
                     'Early Education', 'Learning Resources',
                     'Physical Education Equipment', 'Faux Plants and Trees',
                     'Desk Parts', 'Serving Dishes & Platters', 'Water Filter Pitchers',
                     'Shower Curtain Rods', 'Table Accessories',
                     'Sandboxes & Sand Toys', 'Meeting & Collaborative Spaces',
                     'Desktop Organizers & Desk Pads',
                     'Napkin Rings, Place Card Holders & Food Markers',
                     'Partition & Panel Hardware Accessories', 'Cash Handling', 'Hooks',
                     'Novelty Lighting', 'Protection Plans',
                     'Stages, Risers and Accessories']

def search_wayfair_bm25(keywords: str,
                        category: Optional[Categories] = None,
                        top_k: int = 5,
                        agent_state=None
                        ) -> list[dict[str, Union[str, int, float]]]:
    """Search the wayfair home goods + furniture catalog, get top_k results ranked by BM25

    This is direct keyword search along with optional category filtering.

    Args:
        keywords: The search query string.
        category: category to filter products by, unfiltered when not present
        top_k: The number of top results to return.

    Returns:
        Search results as a list of dictionaries with 'id', 'title', 'description', 'category', and 'score' keys.

    """

    print("search", keywords)
    required_keywords = [term[1:] for term in keywords.split() if term.startswith("+")]
    bm25_scores = np.zeros(len(corpus))
    for term in snowball_tokenizer(keywords):
        bm25_scores += corpus['title_snowball'].array.score(term) * 10
        bm25_scores += corpus['description_snowball'].array.score(term) * 1

    # for required_term in snowball_tokenizer(" ".join(required_keywords)):
    #     required_score = (corpus['title_snowball'].array.score(required_term) +
    #                       corpus['description_snowball'].array.score(required_term))
    #     bm25_scores[required_score == 0] = 0
    # Filter by category
    if category:
        print("Filtering by category:", category)
        cat_tokenized = snowball_tokenizer(category)
        category_mask = corpus['category_snowball'].array.score(cat_tokenized) > 0
        bm25_scores = bm25_scores * category_mask


    top_k_indices = np.argsort(bm25_scores)[-top_k:][::-1]
    bm25_scores = bm25_scores[top_k_indices]
    top_movies = corpus.iloc[top_k_indices].copy()
    top_movies.loc[:, 'score'] = bm25_scores

    results = []
    for id, row in top_movies.iterrows():
        results.append({
            'id': row['doc_id'],
            'title': row['title'],
            'description': row['description'],
            'category': row['category'],
            'score': row['score']
        })
    return results



search_wayfair_bm25("geometric style couch", top_k=5)

search geometric style couch


[{'id': 6346,
  'title': 'kaat 3 - light candle style geometric chandelier',
  'description': 'welcome guests to your home with a splash of statement lighting , illuminate your bedroom or light up your dining room table with this charismatic geometric chandelier . made from steel in a handsome metallic finish , this alluring design showcases a simple circular canopy , a straight down rod , and a distinctive openwork geometric frame around a contemporary candelabra-style base . this hardwired modern luminary accommodates three lightbulbs of up to 60 w each ( bulbs not included ) .',
  'category': 'Lighting',
  'score': 45.974431455135345},
 {'id': 39358,
  'title': 'deloris 4 - light candle style geometric chandelier',
  'description': 'this 4-light pendant features a unique design that enhances the contemporary . it also adds a modern style atmosphere to your home for a more fashionable feel .',
  'category': 'Lighting',
  'score': 43.722705125808716},
 {'id': 22831,
  'title': 'matild

In [ ]:
from cheat_at_search.agent.pydantize import make_tool_adapter

search_tool = make_tool_adapter(search_wayfair_bm25)

tool_info = {search_wayfair_bm25.__name__: search_tool}
tool_info

{'search_wayfair_bm25': (cheat_at_search.agent.pydantize.Search_wayfair_bm25Args,
  {'type': 'function',
   'name': 'search_wayfair_bm25',
   'description': "Search the wayfair home goods + furniture catalog, get top_k results ranked by BM25\n\n    This is direct keyword search along with optional category filtering.\n\n    Args:\n        keywords: The search query string.\n        category: category to filter products by, unfiltered when not present\n        top_k: The number of top results to return.\n\n    Returns:\n        Search results as a list of dictionaries with 'id', 'title', 'description', 'category', and 'score' keys.",
   'parameters': {'properties': {'keywords': {'title': 'Keywords',
      'type': 'string'},
     'category': {'anyOf': [{'enum': ['Furniture',
         'Kitchen & Tabletop',
         'Browse By Brand',
         'Home Improvement',
         'Décor & Pillows',
         'Outdoor',
         'Storage & Organization',
         'Bed & Bath',
         'Baby & Kids'

In [ ]:
def call_tool(tool_info, item, agent_state) -> dict:

    # Lookup how the agent wants to call the tool
    tool_name = item.name
    tool = tool_info[tool_name]
    ToolArgsModel = tool[0]
    tool_fn = tool[2]
    fn_args: ToolArgsModel = ToolArgsModel.model_validate_json(item.arguments)

    print(f"Calling {tool_name} with args {fn_args}")
    # The tool call function itself (ie search)
    # wrapped in something helping with serialization
    py_resp, json_resp = tool_fn(fn_args, agent_state=agent_state)
    print("output", py_resp)

    # 4. Provide function call results to the model
    return {
        "type": "function_call_output",
        "call_id": item.call_id,
        "output": json_resp,
    }


## Setup agentic loop

Setup and run agentic loop within a minimal harness

In [ ]:
import textwrap
from time import sleep


system_prompt = """
    You take user search queries and use a search tool to find furniture / home goods products.

    Look at the search tools you have, their limitations, how they work, etc when forming your plan.

    Finally return results to the user per the SearchResults schema, ranked best to worst.

    Gather results until you have 10 best matches you can find. It's important to return at least 10.

    Consider possibly

    * Not searching categories if no relevant results found

    It's very important you consider carefully the correct ranking as you'll be evaluated on
    how close that is to the average furniture shoppers ideal ranking.

    Here are some examples of products and relevant / irrelevant results

"""



class SearchResults(BaseModel):
    """The ranked, top 10 search results ordered most relevant to least."""
    results_summary: str = Field(description="The message from you summarizing what you found")

    ranked_results: list[int] = Field(description="Top ranked search results (their doc_ids)")


def agent_run(tool_info,
              text_format,
              inputs,
              model='gpt-5-nano',
              agent_state=None,
              summary=True):

    tool_calls = True
    resp = None
    while tool_calls:
        failing = True
        num_failures = 0
        while failing:
            try:
                # print(inputs)
                resp = openai.responses.parse(
                    model=model,
                    input=inputs,
                    tools=[tool[1] for tool in tool_info.values()],
                    reasoning={
                        "effort": "medium",
                        "summary": "auto" if summary else "none"
                    },
                    text_format=text_format
                )
                failing = False
            except Exception:
                failing = True
                num_failures += 1
                if num_failures > 3:
                    raise
                sleep(1)
        inputs += resp.output
        if summary:
            usage = resp.usage
            print("--")
            print(f"InpTok: {usage.input_tokens}")
            print(f"OutTok: {usage.output_tokens}")
            for item in resp.output:
                if item.type == "reasoning":
                    print("Reasoning:")
                    for summary_item in item.summary:
                        print(textwrap.fill(summary_item.text, 80), "\n")
                    item.summary = []

        for item in resp.output:
            tool_calls = False
            if item.type == "function_call":
                tool_calls = True
                # *** Get the tool, and package
                # up the call to the tool (our python function)
                tool_response = call_tool(tool_info, item,
                                          agent_state=agent_state)

                # 4. Provide function call results to the model
                inputs.append(tool_response)
    return resp, inputs



## Wrap in a baby harness

Create a little search harness to issue searches and return results.

In [ ]:

def _error_msg(error):
    print(error)
    return {"role": "user", "content": f"Oh this isn't good, it turns out: {error}. Please try again"}



def search(tools=None,
           inputs=None,
           agent_state=None,
           model="gpt-5"):
    """A little search harness."""
    error = True
    resp = None
    if tools is None:
        tools = [search_wayfair_bm25]
    tool_info = {}
    for tool in tools:
        search_tool = make_tool_adapter(tool)
        tool_info[tool.__name__] = search_tool
    while error:
        resp, inputs = agent_run(tool_info,
                                 text_format=SearchResults,
                                 inputs=inputs,
                                 model="gpt-5-mini")
        # Validate what came back, ensure it fits
        # Lookup each doc id in the response
        num_results = len(resp.output_parsed.ranked_results)
        error = False

    return resp.output_parsed

query = "small woven pouf"
inputs = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": query}]
resp = search(inputs=inputs)
resp

--
InpTok: 757
OutTok: 173
Reasoning:
**Searching for poufs**  I need to call the search function for a "small woven
pouf." The category might be "Décor & Pillows" or "Furniture." I'll set the
top_k to 20, even though the default is 5, to make sure I get at least 10
results. Since I only need one tool call, I won't bother with
multi_tool_use.parallel. So, I'll go ahead with using
functions.search_wayfair_bm25 with those keywords and category. Let's see what I
find! 

Calling search_wayfair_bm25 with args keywords='small woven pouf' category='Décor & Pillows' top_k=20
search small woven pouf
Filtering by category: Décor & Pillows
output [{'id': 31076, 'title': 'small picture frame', 'description': 'this beautiful photo frame makes the perfect finishing touch to your favorite photos and prints . with a crisp white finish and durable polystyrene construction , you can proudly display your favorite logophilia print in chic style .', 'category': 'Décor & Pillows', 'score': 36.65983200073242

SearchResults(results_summary='Searched for "small woven pouf" and returned the 10 best matching pouf/ottoman results (hand‑woven, jute/seagrass/cotton styles) ranked most to least relevant.', ranked_results=[19532, 38619, 38618, 38620, 38617, 38616, 9800, 50, 2070, 23333])

## Compare results to just BM25

Compare a set of queries to BM25 baseline

In [ ]:
from cheat_at_search.strategy import SearchStrategy
from cheat_at_search.search import run_strategy
from cheat_at_search.wands_data import judgments

class AgenticSearchStrategy(SearchStrategy):
    def __init__(self,
                 corpus,
                 workers=1,
                 model="gpt-5-mini",
                 system_prompt=system_prompt,
                 tools=None):
        self.tools = tools
        self.model=model
        super().__init__(corpus, workers=workers)

    def search(self, query, k):
        agentic_query = "Find me: " + query
        print("_________---___________")
        print(agentic_query)
        print("_________---___________")
        inputs = [{"role": "system", "content": system_prompt},
                  {"role": "user", "content": agentic_query}]
        resp = search(tools=self.tools,
                      inputs=inputs,
                      model=self.model)
        return (resp.ranked_results,
                [1.0] * len(resp.ranked_results))

# Get 20 random queries from judgments
seed = 1234
np.random.seed(seed)
random_queries = np.random.choice(judgments['query'].unique(), 100)
selected_judgments = judgments[judgments['query'].isin(random_queries)]
selected_judgments

strategy_agentic_bm25 = AgenticSearchStrategy(corpus, workers=4, model="gpt-5", tools=[search_wayfair_bm25])
graded_agentic_bm25 = run_strategy(strategy_agentic_bm25, selected_judgments)

Searching:   0%|          | 0/88 [00:00<?, ?it/s]

_________---___________
Find me: turquoise pillows
_________---___________
_________---___________
Find me: beds that have leds
_________---___________
_________---___________
Find me: blk 18x18 seat cushions
_________---___________
_________---___________
Find me: chrome bathroom 4 light vanity light
_________---___________
--
InpTok: 758
OutTok: 148
Reasoning:
**Searching for pillows**  I need to use the search tool to look for "turquoise
pillows." The goal is to gather at least 10 results. The default setting returns
five results, but I think I could set it to 50 to ensure I have enough. I don’t
think I need to use multiple tools for this task. So, I'll call
functions.search_wayfair_bm25 with the keywords "turquoise pillows" and a top_k
of 50 to ensure I get a solid list. 

Calling search_wayfair_bm25 with args keywords='turquoise pillows' category=None top_k=50
search turquoise pillows
output [{'id': 5998, 'title': 'noriega turquoise geometric pillow', 'description': 'make a bold s

Searching:   1%|          | 1/88 [00:10<15:26, 10.65s/it]

--
InpTok: 3367
OutTok: 346
Reasoning:
_________---___________
Find me: 3 1/2 inch drawer pull
_________---___________
--
InpTok: 3595
OutTok: 338
Reasoning:
**Compiling LED bed results**  I’ve returned multiple results for "beds that
have LEDs," and I need to compile the top 10 ranked matches. My priority is
finding actual beds with LED features. The top relevant items are IDs 37143,
37142, and 37141—all variations of upholstered LED storage platform beds.
There's also ID 29434, a Febus platform bed with a lighted headboard. I also
have to be cautious about items like those that mention turbocharger surface
mounts or are unrelated, focusing only on the LED beds. 

**Searching for more LED beds**  I need to find more beds with LED features, so
I’ll explore additional keywords like "LED bed frame," "bed with lights
headboard," "LED platform bed storage," and "LED upholstered bed." It might be
beneficial to use multi_tool_use.parallel. I can call
functions.search_wayfair_bm25 directly mu

Searching:   2%|▏         | 2/88 [00:13<08:19,  5.81s/it]

...

Searching:  99%|█████████▉| 87/88 [07:37<00:04,  4.20s/it]

--
InpTok: 2046
OutTok: 391
Reasoning:


Searching: 100%|██████████| 88/88 [07:47<00:00,  5.80s/it]

--
InpTok: 9922
OutTok: 1547
Reasoning:
**Finding Nespresso options**  I’ve found several relevant products related to
the user's request for the "Nespresso Vertuo Next Premium by Breville with
Aeroccino." It looks like I need to narrow down the top 10 matches from the
search results. For instance, there’s the Nespresso Breville Vertuo coffee
maker, which seems like a strong option. I’m checking other bundles too,
including those with Aeroccino, to ensure I capture what the user is looking
for. I’ll rank them accordingly! 

**Ranking Nespresso options**  I'm focusing on finding the best matches for the
user's query about the "Vertuo Next Premium by Breville with Aeroccino." I need
to see which items mention "Vertuo Next." The top contender seems to be product
id 37985, which includes the features I'm looking for, like recycled materials
and connectivity options. I’ll check for other options, including any bundles
with Aeroccino, but I understand the user's request specifically includes

Searching: 100%|██████████| 88/88 [07:48<00:00,  5.32s/it]


In [ ]:
from cheat_at_search.search import ndcgs, graded_bm25, vs_ideal
# 5,5,4,7
ndcgs(graded_agentic_bm25).sort_index(), ndcgs(graded_agentic_bm25).mean()

(query
 3 1/2 inch drawer pull                      0.749627
 3/4 size mattress                           0.228695
 46 inch closet door                         0.314366
 48 inch bathroom vanity with trough sink    0.333333
 48 inch kitchen hood                        1.000000
                                               ...   
 whimsical solar lights                      0.783137
 window wall accent                          1.000000
 wishbone chair                              0.560945
 writing desk                                1.000000
 zodiac pillow                               0.715486
 Name: ndcg, Length: 86, dtype: float64,
 np.float64(0.5873392714795688))

In [ ]:
ndcgs(graded_bm25[graded_bm25['query'].isin(random_queries)]).sort_index(), ndcgs(graded_bm25[graded_bm25['query'].isin(random_queries)]).mean()

(query
 3 1/2 inch drawer pull                      0.578648
 3/4 size mattress                           0.128031
 46 inch closet door                         0.307727
 48 inch bathroom vanity with trough sink    0.333333
 48 inch kitchen hood                        0.951226
                                               ...   
 whimsical solar lights                      0.863433
 window wall accent                          1.000000
 wishbone chair                              0.674750
 writing desk                                1.000000
 zodiac pillow                               0.829291
 Name: ndcg, Length: 88, dtype: float64,
 np.float64(0.5203953229627256))

## Measure embedding search w/ agents... and combined w/ BM25

In [ ]:
from cheat_at_search.wands_data import product_embeddings
from sentence_transformers import SentenceTransformer, util


minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


def search_wayfair_embeddings(keywords: str,
                              category: Optional[Categories] = None,
                              top_k: int = 5
                              ) -> list[dict[str, Union[str, int, float]]]:
    """Search the wayfair home goods + furniture catalog, get top_k results ranked by minilm of product name + description

    This is an embedding search along with optional category filtering.

    Args:
        keywords: The search query string to embed into minilm, and search product name+description
        category: category to filter products by, unfiltered when not present
        top_k: The number of top results to return.

    Returns:
        Search results as a list of dictionaries with 'id', 'title', 'description', 'category', and 'score' keys.

    """
    query_embedded = minilm.encode(keywords)
    # print(query_embedded)

    # for required_term in snowball_tokenizer(" ".join(required_keywords)):
    #     required_score = (corpus['title_snowball'].array.score(required_term) +
    #                       corpus['description_snowball'].array.score(required_term))
    #     bm25_scores[required_score == 0] = 0
    # Filter by category
    if category:
        print("Filtering by category:", category)
        cat_tokenized = snowball_tokenizer(category)
        category_mask = corpus['category_snowball'].array.score(cat_tokenized) > 0

    similarity_scores = np.dot(product_embeddings, query_embedded)
    if category:
        similarity_scores = similarity_scores * category_mask

    top_k_indices = np.argsort(similarity_scores)[-top_k:][::-1]
    similarity_scores = similarity_scores[top_k_indices]
    top_movies = corpus.iloc[top_k_indices].copy()
    top_movies.loc[:, 'score'] = similarity_scores

    results = []
    for id, row in top_movies.iterrows():
        results.append({
            'id': row['doc_id'],
            'title': row['title'],
            'description': row['description'],
            'category': row['category'],
            'score': row['score']
        })
    return results

search_wayfair_embeddings("geometric style couch", top_k=5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[{'id': 27548,
  'title': "jarrett 112 '' wide sofa & chaise",
  'description': "gathering everyone for a movie marathon , or hosting a game night ? whatever your living room needs , it 's hard to beat sectional sofas when it comes to giving everyone a seat ( and your living room an on-trend look ) . take this one , for example , ideal for anchoring your space with a touch of mid-century-inspired style , this piece features a streamlined silhouette , recessed arms , and dowel feet . crafted with a solid wood frame , this piece features foam and fiber filling for an inviting feel , and is wrapped in neutral polyester blend upholstery that allows it o easily join a variety of color schemes .",
  'category': 'Furniture',
  'score': 0.7120468020439148},
 {'id': 42254,
  'title': "hillcrest 85 '' linen square arm sofa bed",
  'description': 'the classic design of this sofa gives it a distinctive look that elevates your well-curated collection . the back cushion can be folded down so that th

In [ ]:
class EmbeddingStrategy(SearchStrategy):
    """Just naive search of minilm embeddings"""
    def __init__(self,
                 corpus,
                 workers=1):
        super().__init__(corpus, workers=workers)

    def search(self, query, k):
        resp = search_wayfair_embeddings(query, top_k=k)
        ids = [r['id'] for r in resp]
        scores = [r['score'] for r in resp]
        return (ids, scores)

strategy_embedding = EmbeddingStrategy(corpus, workers=4)
graded_embedding = run_strategy(strategy_embedding, selected_judgments)

Searching: 100%|██████████| 88/88 [00:10<00:00,  8.78it/s]


In [ ]:
ndcgs(graded_embedding).mean()

np.float64(0.5140830469715443)

In [ ]:
inputs = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": "small woven pouf"}]
resp = search(inputs=inputs, tools=[search_wayfair_embeddings, search_wayfair_bm25])
resp

--
InpTok: 1263
OutTok: 391
Reasoning:
Calling search_wayfair_embeddings with args keywords='small woven pouf' category='Décor & Pillows' top_k=15
Filtering by category: Décor & Pillows
output [{'id': 12353, 'title': 'kazi jacquard pom pom cotton throw pillow', 'description': 'adding both comfort and artful appeal to your ensemble , pillows are perfect for just about any area of your abode . this solid dec pillow features small all-over pom tufts , creating a beautiful texture and fresh tonal pattern . while the 100 % cotton fabric provides a natural touch for a casual and modern update . finished with a hidden zipper closure and removable woven polyester filling for easy care ; makes this pillow an easy stylish addition to any space ! this pillow is also oeko-tex certified , meaning it does not contain any harmful substances or chemicals , ensuring quality comfort and wellness . complete the set with coordinate valance , panel , and bedding - available and sold separately .', 'categor

SearchResults(results_summary='Searched the Wayfair catalog for "small woven pouf" using both BM25 and embedding search. The BM25 keyword search returned the most relevant pouf/ottoman results; embedding results returned mostly pillows and decor, so I ranked the top BM25 matches. I’m returning the 10 best-matching poufs (ordered most to least relevant).', ranked_results=[19532, 38423, 21458, 38620, 38619, 38618, 38617, 38616, 38895, 9800])

In [ ]:
strategy = AgenticSearchStrategy(corpus, workers=4, tools=[search_wayfair_embeddings])
graded_agentic_emb = run_strategy(strategy, selected_judgments)

Searching:   0%|          | 0/88 [00:00<?, ?it/s]

_________---___________
Find me: turquoise pillows
_________---___________
_________---___________
Find me: beds that have leds
_________---___________
_________---___________
Find me: blk 18x18 seat cushions
_________---___________
_________---___________
Find me: chrome bathroom 4 light vanity light
_________---___________
--
InpTok: 772
OutTok: 150
Reasoning:
**Searching for turquoise pillows**  The user wants to search for "turquoise
pillows," so I need to use the search function related to Wayfair and return the
top 10 matches. The top_k parameter typically defaults to 5, but just to be
safe, I should set it to 50 to ensure I gather enough options. I see that "Décor
& Pillows" is a valid category, so I’ll go ahead and call the function with that
category and the adjusted top_k value. 

Calling search_wayfair_embeddings with args keywords='turquoise pillows' category='Décor & Pillows' top_k=50
Filtering by category: Décor & Pillows
--
InpTok: 777
OutTok: 167
Reasoning:
**Searching 

Searching:   1%|          | 1/88 [00:11<16:19, 11.25s/it]

--
InpTok: 3103
OutTok: 329
Reasoning:
**Preparing search results JSON**  I have 20 results, and I need to return the
top 10 ranked from best to worst. I'll create a SearchResults JSON that includes
a summary and a ranked list of document IDs. First, I’ll sort the results by
score in descending order. The top 10 scores are: 32459 (0.7777), 32658
(0.7559), and so on. I want a concise summary, maybe something like “Found 20
lighting results; top 10 4-light chrome vanity lights are listed.” I'll make
sure to adhere to the correct schema! 

_________---___________
Find me: 3 1/2 inch drawer pull
_________---___________


Searching:   2%|▏         | 2/88 [00:12<07:55,  5.53s/it]

...

Searching:  99%|█████████▉| 87/88 [08:23<00:05,  5.13s/it]

--
InpTok: 12133
OutTok: 1205
Reasoning:
**Selecting top vanity items**  I need to find the best matches for "48 inch
bathroom vanity with trough sink." I'll sift through the results to rank the top
10 based on their relevance. I found several items: there's the "trough 48''
single bathroom vanity top" and "trough concrete rectangular bathroom sink." I
see that the "48'' single bathroom vanity top with sink" scores the highest, and
I'll also consider the "48'' single sink bathroom vanity set" and a few others
that mention 48 inches. 

**Choosing relevant vanities**  I need to decide if I should include doubles
among the items, but I want to prioritize 48" single vanities with trough or
integrated trough style sinks. The 'trough' items definitely seem the best bet!
I’ll provide at least 10 relevant IDs, focusing on those that match 48 inches
and a trough sink closely. I’ll use their scores as guidance, starting with the
highest relevance item, like the "48'' single bathroom vanity top w

Searching: 100%|██████████| 88/88 [08:27<00:00,  4.88s/it]

--
InpTok: 3079
OutTok: 1516
Reasoning:
**Ranking results**  I have 20 results to work with and need to return the top
10 matches ranked by their scores. The tool provides items with IDs and scores,
so I’ll select the top 10 based on those scores. I'll create a list of integers
for the ranked results, and also put together a summary string to summarize what
I found. The highest scores include specific products like a 48-inch island
range hood and various copper hood models. 

**Selecting best matches**  I have additional results to consider. Among them,
several products are specifically 48 inches, like a professional stainless
ducted wall mount range hood and a chimney range hood. Given that the user asked
for a "48 inch kitchen hood," I need to prioritize those results. I'll ensure
that the top matches include these specific products, as they align with the
user's request. So, I’ll carefully rank the best matches accordingly,
considering the relevant score and size. 

**Identifying 48

Searching: 100%|██████████| 88/88 [08:28<00:00,  5.78s/it]


In [ ]:
ndcgs(graded_agentic_emb).mean()

np.float64(0.5446005920196408)

In [ ]:
strategy_combined = AgenticSearchStrategy(corpus, workers=4, tools=[search_wayfair_embeddings, search_wayfair_bm25])
graded_agentic_combined = run_strategy(strategy_combined, selected_judgments)

Searching:   0%|          | 0/88 [00:00<?, ?it/s]

_________---___________
Find me: turquoise pillows
_________---___________
_________---___________
Find me: beds that have leds
_________---___________
_________---___________
Find me: blk 18x18 seat cushions
_________---___________
_________---___________
Find me: chrome bathroom 4 light vanity light
_________---___________
--
InpTok: 1269
OutTok: 232
Reasoning:
Calling search_wayfair_embeddings with args keywords='black 18x18 seat cushions' category='Furniture Cushions' top_k=15
Filtering by category: Furniture Cushions
output [{'id': 40135, 'title': 'sunbrella bench cushion', 'description': '', 'category': 'Furniture Cushions', 'score': 0.5314621925354004}, {'id': 14345, 'title': 'replacement wood vinyl padded seat cushion', 'description': 'are you an event planner , an event facility manager or an event rental company ? when you have furniture that is used frequently , eventually items need replacing . if the upholstery on your wood folding chairs are torn , worn looking or discolo

Searching:   1%|          | 1/88 [00:22<31:55, 22.02s/it]

--
InpTok: 4326
OutTok: 675
Reasoning:
**Combining search results**  I need to combine two sets of search results and
rank them from best to worst, aiming for the top 10 while managing some
duplicates. The final output must include a "results_summary" string and a list
of document IDs. I want results that specifically match keywords like "chrome,"
"4-light," and "vanity light" in the lighting category. After evaluating the
BM25 scores, I'll select the top 10 most relevant IDs: 32459, 32658, 17491,
19632, 37524, 37528, 20885, 15892, 33907, and 10259. 

**Organizing search results**  I noticed that embeddings had a slightly
different order but were generally similar. To align with user expectations, I
should use BM25 ranking based on keyword matches for searches. I'll return the
top ten IDs in that order and include a brief summary message stating I've found
these 10 best matches. The results will summarize the findings concisely and
list any key features while keeping it minimal since t

Searching:   2%|▏         | 2/88 [00:23<14:22, 10.03s/it]

...

Searching:  99%|█████████▉| 87/88 [10:26<00:05,  5.46s/it]

--
InpTok: 6140
OutTok: 553
Reasoning:
**Processing ranked results**  I’ve received many results and need to return the
top 10 based on shopping preference. The tool sorted them by score, so I just
need to select the 10 best matches from that list. The query "rack glass" might
refer to either a "glass rack" for dishwashing or a "wine glass rack" for home
use. The most relevant items seem to be opticlean glass racks followed by
stainless steel and wall-mounted options. I’ll ensure I format my response
correctly with the required criteria. 

**Selecting top items**  I’m looking at the IDs from my results, focusing on the
top matches: several opticlean options and a stainless steel glass rack, along
with a few wine glass racks. I should consider swapping out some extender racks
for other wall-mounted shelves, like the catton solid wood option, which fits
home use. Given that the user’s query was ambiguous, I believe including both
commercial and home products is best. I'll summarize my fi

Searching: 100%|██████████| 88/88 [10:27<00:00,  4.38s/it]

--
InpTok: 9656
OutTok: 1221
Reasoning:
**Selecting bathroom vanities**  I'm looking at several relevant items,
including "trough 48'' single bathroom vanity top" (ID 41393) and "trough
concrete rectangular bathroom sink" (ID 41395), among other 48" vanities. Since
the user wants a "48 inch bathroom vanity with trough sink," I'm prioritizing
items with trough sinks or full vanity sets that include an integrated trough
sink. The top two ranked items specifically mention troughs and meet the 48"
criteria. I’ll scan the results for more matches with trough or integrated sink
mentions to find the best ten. 

**Compiling 48" vanity options**  The user is looking for a 48-inch vanity with
a trough sink, likely a single vanity with a long rectangular trough. I’ll
prioritize compatible items like the "vanity top with trough sink" (ID 41393)
and "trough concrete sink" (ID 41395). I’ll also consider vanity sets with a
single long sink. Items like ID 39029, which is a "48'' single bathroom vanity

Searching: 100%|██████████| 88/88 [10:28<00:00,  7.14s/it]


In [ ]:
ndcgs(graded_agentic_combined).mean()

np.float64(0.6035485254933838)

## Create a strategy with feedback

Some experiments where Doug embarasses himself.

### Pretend we have a subagent that knows the right answer

This could be simulating a reranker, a model trained to classify relevant from irrelevant, or really anything.

### Use skills to tweak retrieval strategy

Certain classifications queries should use different tools, or use them differently

## Force the agent to retry if its results degrade

Add a guardrail to prevent result quality from degrading.

## What else?

* How could we change the retrieval tool?
* Does it help to improve ranking of the tool?

(Your ideas here)


In [ ]:

system_prompt = """
    You take user search queries and use a search tool to find furniture / home goods products.

    Look at the search tools you have, their limitations, how they work, etc when forming your plan.

    Finally return results to the user per the SearchResults schema, ranked best to worst.

    Gather results until you have 10 best matches you can find. It's important to return at least 10.

    You'll get feedback from the user on whether results satisfies them:

    ☹️ - ACTIVELY FRUSTRATES USER, do not include unless absolutely necesarry
    😑 - Meh results. OK in a pinch. But there could be better.
    😃 - Solves users problem. Good job! Rank this highest

    If you get this feedback, try again to:

    * Find results that can replace the ☹️ / 😑 with 😃
    * Hypothesize new queries that might find 😃
    * Use the content in known 😃 results to create new queries to the search tool

    Consider possibly

    * Not searching categories if no relevant results found

    It's very important you consider carefully the correct ranking as you'll be evaluated on
    how close that is to the average furniture shoppers ideal ranking.

"""

In [ ]:
from cheat_at_search.wands_data import labeled_query_products

def _grade_to_emoji(grade):
    if grade == 0:
        return '☹️'
    elif grade == 1:
        return '😑'
    elif grade == 2:
        return '😃'
    return '☹️'


class SearchResult(BaseModel):
    """A search result and your best guess at relevance."""
    doc_id: int = Field(description="The doc id of the search result")
    grade: Literal['☹️', '😑', '😃'] = Field(description="How relevant this is to the query, in your estimation")


class SearchResults(BaseModel):
    """The ranked, top 10 search results ordered most relevant to least."""
    results_summary: str = Field(description="The message from you summarizing what you found")

    ranked_results: list[SearchResult] = Field(description="Ranked search results")


def grades(query, search_results: SearchResults):
    """Get array of grades for each search result."""
    query_judgments = labeled_query_products[labeled_query_products['query'] == query]
    r_value = []
    for search_result in search_results.ranked_results:
        doc_id = search_result.doc_id
        title = corpus[corpus['doc_id'] == doc_id]['title'].iloc[0]
        doc_judgments = query_judgments[query_judgments['doc_id'] == doc_id]
        if len(doc_judgments) == 0:
            r_value.append((title, doc_id, _grade_to_emoji(None)))
        else:
            r_value.append((title, doc_id, _grade_to_emoji(int(doc_judgments['grade'].values[0]))))
    return r_value

def count_smileys(gradeds):
    count = 0
    for graded in gradeds:
        if graded[2] == '😃':
            count += 1
    return count


def degrade_hook_check(query):
    def search_degrade_hook(resp, inputs):
        """Check outputs, if results are worse, then fail!"""
        search_results = resp
        # For every past output
        all_graded = []
        for input in inputs:
            if hasattr(input, 'content') and input.content is not None and hasattr(input.content[-1], 'parsed'):
                result = input.content[-1].parsed
                if isinstance(result, SearchResults):
                    all_graded.append(grades(query, result))
        if len(all_graded) > 1:
            last_smileys = count_smileys(all_graded[-2])
            current_smileys = count_smileys(all_graded[-1])
            if last_smileys > current_smileys:
                inputs.append(_error_msg(f"You've degraded your relevance, previously found {last_smileys} relevant results , and now found {current_smileys}"))
                return False
        return True
    return search_degrade_hook

In [ ]:
def search_wayfair_embeddings_guarded(keywords: str,
                                      category: Optional[Categories] = None,
                                      top_k: int = 5,
                                      agent_state=None
                                      ) -> list[dict[str, Union[str, int, float]]]:
    """Search the wayfair home goods + furniture catalog, get top_k results ranked by minilm of product name + description

    This is an embedding search along with optional category filtering.

    Args:
        keywords: The search query string to embed into minilm, and search product name+description
        category: category to filter products by, unfiltered when not present
        top_k: The number of top results to return.

    Returns:
        Search results as a list of dictionaries with 'id', 'title', 'description', 'category', and 'score' keys.

    """
    if not agent_state:
        print(f"Calling emb_g {keywords} -- {category} -- {top_k}")
        return search_wayfair_embeddings(keywords, category, top_k)

    func_name = 'search_wayfair_embeddings_guarded'
    q_guard = []

    if func_name not in agent_state['past_queries']:
        agent_state['past_queries'][func_name] = q_guard
    else:
        q_guard = agent_state['past_queries'][func_name]

    if keywords in q_guard:
        err_msg = "Error! You've already tried query: " + keywords + " Be more creative and explore more!"
        print(func_name, err_msg)
        return err_msg
    else:
        q_guard.append(keywords)
        agent_state['past_queries'][func_name] = q_guard
        print(f"Calling emb_g {keywords} -- {category} -- {top_k}")
        return search_wayfair_embeddings(keywords, category, top_k)


def search_wayfair_bm25_guarded(keywords: str,
                                category: Optional[Categories] = None,
                                top_k: int = 5,
                                agent_state=None
                                ) -> list[dict[str, Union[str, int, float]]]:
    """Search the wayfair home goods + furniture catalog, get top_k results ranked by BM25

    This is direct keyword search along with optional category filtering.

    Queries will be rejected if you've already tried a query for this tool.

    Args:
        keywords: The search query string.
        category: category to filter products by, unfiltered when not present
        top_k: The number of top results to return.

    Returns:
        Search results as a list of dictionaries with 'id', 'title', 'description', 'category', and 'score' keys.

    """
    if not agent_state:
        print(f"Calling bm25_g {keywords} -- {category} -- {top_k}")
        return search_wayfair_bm25(keywords, category, top_k)

    func_name = 'search_wayfair_bm25_guarded'
    q_guard = []
    if func_name not in agent_state['past_queries']:
        agent_state['past_queries'][func_name] = q_guard
    else:
        q_guard = agent_state['past_queries'][func_name]

    if keywords in q_guard:
        err_msg = "Error! You've already tried query: " + keywords + " Be more creative and explore more!"
        print(func_name, err_msg)
        return err_msg
    else:
        print(f"Calling bm25_g {keywords} -- {category} -- {top_k}")
        q_guard.append(keywords)
        agent_state['past_queries'][func_name] = q_guard
        return search_wayfair_bm25(keywords, category, top_k)

In [ ]:
agent_state = {'past_queries': {}}
search_wayfair_bm25_guarded("geometric style couch", top_k=5, agent_state=agent_state)
search_wayfair_bm25_guarded("geometric style couch", top_k=5, agent_state=agent_state)

Calling bm25_g geometric style couch -- None -- 5
search geometric style couch
search_wayfair_bm25_guarded Error! You've already tried query: geometric style couch Be more creative and explore more!


"Error! You've already tried query: geometric style couch Be more creative and explore more!"

In [ ]:

class AgenticSearchStrategyRalphed(SearchStrategy):
    def __init__(self,
                 corpus,
                 workers=1,
                 model="gpt-5",
                 system_prompt=system_prompt,
                 skills=None,
                 tools=None):
        self.tools = tools
        self.model=model
        self.skills = skills
        super().__init__(corpus, workers=workers)

    def _inject_skill_on_kw(self, query):
        skill_prompts = []
        if self.skills:
            for skill_rule, skill_prompt in self.skills.items():
                if skill_rule in query:
                    skill_prompts.append(skill_prompt)
        return skill_prompts

    def search(self, query, k):
        validator = degrade_hook_check(query)

        agentic_query = "Find me: " + query
        print("_________---___________")
        print(agentic_query)
        print("_________---___________")
        inputs = [{"role": "system", "content": system_prompt},
                  {"role": "user", "content": agentic_query}]

        skill_prompts = self._inject_skill_on_kw(query)
        for prompt in skill_prompts:
            print(f"Using skill: {prompt}")
            inputs.append({"role": "user", "content": prompt})

        tries = 0
        agent_state = {'past_queries': {}}
        while True:
            print("********")
            print(f"ROUND {tries}")
            valid = False
            resp = None
            while not valid:
                resp = search(inputs=inputs,
                            tools=self.tools,
                            model=self.model,
                            agent_state=agent_state)
                valid = validator(resp, inputs)
                if not valid:
                    print(f"Validation check failed!")

            graded = grades(query, resp)
            message_back = f"""
                These results can be improved. Can you look at them and fix them?

                Get creative

                Here's how I feel about the results, please improve them.
                Try to find more 😃 results to replace the 😑 ones.
                Or at least 😑 to replace ☹️

                Return a better ranking, with the happier emojis towards the top

                Reminder the search is for: {query}

            """
            for title, doc_id, emoji in graded:
                message_back += f"{emoji} {title}\n"
            print(message_back)
            inputs.append({"role": "user", "content": message_back})
            tries += 1
            if tries > 3:
                break

        return (resp.ranked_results,
                [1.0] * len(resp.ranked_results))

# kid_prompt = """
# This query mentions kid, toddler, baby, etc try both search tools,
# with and without 'Baby & Kids' as category filter
# """
# fold_out_prompt = """
# If you're asked for a fold out, then, also consider couches
# """
# skills = {'toddler': kid_prompt,
#           'fold out' : fold_out_prompt}

skills = {}

strategy_combined_ralphed = AgenticSearchStrategyRalphed(corpus, workers=4,
                                                         skills=skills,
                                                         tools=[search_wayfair_embeddings_guarded,
                                                                search_wayfair_bm25_guarded])

strategy_combined_ralphed.search("70s inspired furniture", 5)

_________---___________
Find me: 70s inspired furniture
_________---___________
********
ROUND 0
--
InpTok: 1509
OutTok: 408
Reasoning:
**Preparing search results mapping**  I need to ensure that the tools return
lists containing essential fields like id, title, description, category, and
score. My goal is to map this to the SearchResults schema, which means I’ll
create a results summary string and a ranked_results array of up to 10 entries.
Each entry will have a doc_id and a grade represented by emojis to indicate
relevance. I should rank these from best to worst and then call the tools. 

Calling search_wayfair_embeddings_guarded with args keywords='70s inspired furniture' category='Furniture' top_k=10
Calling emb_g 70s inspired furniture -- Furniture -- 10
Filtering by category: Furniture
output [{'id': 25864, 'title': 'karine wardrobe armoire', 'description': 'this tv-armoire magnum opus work is handcrafted beautifully by the artisans which creates an ethnic note . with seven draw

([SearchResult(doc_id=20505, grade='😃'),
  SearchResult(doc_id=34951, grade='😃'),
  SearchResult(doc_id=38894, grade='😃'),
  SearchResult(doc_id=38110, grade='😃'),
  SearchResult(doc_id=34239, grade='😃'),
  SearchResult(doc_id=39720, grade='😃'),
  SearchResult(doc_id=9323, grade='😃'),
  SearchResult(doc_id=16726, grade='😃'),
  SearchResult(doc_id=41355, grade='😃'),
  SearchResult(doc_id=2642, grade='😃')],
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0])

## Run ralphed strategy

In [ ]:
graded_agentic_ralphed = run_strategy(strategy_combined_ralphed, selected_judgments)

Searching:   0%|          | 0/88 [00:00<?, ?it/s]

_________---____________________---___________
Find me: beds that have leds
_________---___________
********
ROUND 0

Find me: turquoise pillows
_________---___________
********
ROUND 0
_________---___________
Find me: blk 18x18 seat cushions
_________---___________
********
ROUND 0
_________---___________
Find me: chrome bathroom 4 light vanity light
_________---___________
********
ROUND 0
--
InpTok: 1508
OutTok: 175
Reasoning:
**Searching for LED beds**  The user is asking to find beds that have LEDs, so I
need to use search tools to get the 10 best matches. I'll start by using
embeddings since that's probably the most effective option. I'll focus on the
"Furniture" category and use keywords like "bed LED," "bed with LED lights,"
"platform bed with lights," and "LED bed frame." The default for the tool is to
return 5 results, but I'll request 10 instead using the right function. 

Calling search_wayfair_embeddings_guarded with args keywords='bed with LED lights' category='Furniture'

Searching:   0%|          | 0/88 [01:13<?, ?it/s]


Streaming output truncated to the last 5000 lines.
specifically looking for "bathroom lighting" and specifying a top_k of 10. After
getting the results, I’ll organize them according to the required schema,
including a summary and a ranked array with grades for relevance. Then I can
proceed to map the doc_ids correctly for the responses. 

**Ranking lighting results**  I'll focus on the "Lighting" category and set
top_k to 10 for each search. Once I gather the results, I need to unify them,
eliminating duplicates by their IDs. For ranking, I can average the scores from
both tools, but I realize their scales might differ, so I'll prefer items with
the category "Lighting" and those containing keywords in their titles or
descriptions. Ultimately, I’ll treat higher scores as better while calling the
parallel tools for efficiency. 

Calling search_wayfair_bm25_guarded with args keywords='bathroom lighting' category='Lighting' top_k=10
Calling bm25_g bathroom lighting -- Lighting -- 10
search

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'SearchResult'

In [ ]:
ndcgs(graded_agentic_ralphed).mean()

NameError: name 'graded_agentic_ralphed' is not defined

In [ ]:
labeled_query_products[labeled_query_products['query'] == "wisdom stone river 3-3/4"].sort_values('grade', ascending=True)

In [ ]:
diff = ndcgs(graded_agentic_combined) - ndcgs(graded_bm25)
diff.dropna().sort_values()

In [ ]:
ndcgs(graded_agentic_combined)